# MACE and NequIP on cuEquivariance: parity with the reference

The two blocks that dominate an equivariant model's cost have a fast path on fused
[cuEquivariance](https://github.com/NVIDIA/cuEquivariance) kernels, selected per evaluation with
`use_fast` (see the *Run Models on Fused GPU Kernels* how-to):

* the message step of the convolution, gather + `uvu` tensor product + sum onto the receivers,
  as one kernel (`xnn.gnn.fast.ConvTensorProduct`, MACE and NequIP);
* MACE's symmetric contraction (the product basis), as one kernel per output irrep.

Both read the model's own parameters through fixed maps (a permutation and per-path factors for
the convolution, a change of coupling basis for the contraction, fitted once in float64), so the
model, its checkpoint and its `state_dict` are unchanged. This notebook checks that the fast paths
reproduce the reference on the systems the models are served on: periodic water at 1 g/cm$^3$
(randomly oriented molecules on a jittered lattice, 64 up to 1728 molecules) for MACE-OFF23 and
NequIP, and diamond silicon up to 4096 atoms for MACE-MP-0, in energy, forces, stress, the
gradients of a force-matching loss (training) and a TorchScript export. Every structure is built
in the notebook.

What to expect: the kernels compute the same function with another summation order, so in
float64 results agree to about $10^{-14}$ relative; in float32 to about $10^{-6}$ (the first fast
evaluation in float32 says so with a `FastPathPrecisionWarning`).

In [1]:
import warnings

warnings.filterwarnings("ignore")          # TorchScript / deprecation chatter of the stack

import copy
import time

import numpy as np
import torch
from ase.build import bulk

from xnn.common.data import structure_to_graph
from xnn.common.models import ForceStressOutput, build_model, from_pretrained
from xnn.common.models import fast
from xnn.common.config import from_dict
from xnn.gnn.fast import AUTO_POLICY, available

assert torch.cuda.is_available() and available(), "needs a CUDA device and xnns[cueq]"
device = torch.device("cuda")
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

NVIDIA A100-SXM4-80GB | torch 2.5.1+cu121


## 1. Systems and helpers

`water(n)` places `n^3` randomly oriented molecules on a jittered cubic lattice 3.104 Å apart
(1 g/cm$^3$); `supercell` tiles the diamond cell of silicon `n x n x n` times with a 0.05 Å random
displacement, so the forces are not zero by symmetry. `run` times one energy + forces + stress evaluation with
the reference or the fast path (best of five, after warm-up); `compare` evaluates both on the same
graph.

In [2]:
SILICON = bulk("Si", "diamond", a=5.431, cubic=True)
MOLECULE = np.array([[0.0, 0.0, 0.119262], [0.0, 0.763239, -0.477047], [0.0, -0.763239, -0.477047]])


def water(n, seed=7):
    rng = np.random.default_rng(seed)
    pos = []
    for i in range(n):
        for j in range(n):
            for k in range(n):
                q, _ = np.linalg.qr(rng.normal(size=(3, 3)))
                q *= np.sign(np.linalg.det(q))
                pos.extend((MOLECULE - MOLECULE[0]) @ q.T + np.array([i, j, k]) * 3.104
                           + rng.normal(scale=0.1, size=3))
    return {"pos": np.array(pos), "atomic_numbers": np.array([8, 1, 1] * n ** 3),
            "cell": np.eye(3) * 3.104 * n, "pbc": [True] * 3}


def supercell(atoms, n, jitter=0.0):
    pos, z, cell = atoms.positions, atoms.numbers, np.asarray(atoms.cell)
    shifts = np.array([[i, j, k] for i in range(n) for j in range(n) for k in range(n)])
    big = (pos[None] + (shifts @ cell)[:, None]).reshape(-1, 3)
    if jitter:
        big = big + np.random.default_rng(n).normal(scale=jitter, size=big.shape)
    return {"pos": big, "atomic_numbers": np.tile(z, len(shifts)), "cell": cell * n, "pbc": [True] * 3}


def graph_of(s, cutoff, dtype):
    s = {**s, "pos": torch.tensor(s["pos"], dtype=dtype), "cell": torch.tensor(s["cell"], dtype=dtype),
         "atomic_numbers": torch.tensor(s["atomic_numbers"])}
    return structure_to_graph(s, cutoff, device=device)


def run(model, g, use_fast, repeats=5):
    fast.set_use_fast(model, use_fast)
    out = model(copy.copy(g))
    for _ in range(2):
        model(copy.copy(g))
    times = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(copy.copy(g))
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return {k: out[k].detach() for k in ("energy", "forces", "stress")}, min(times)


def compare(name, model, s, cutoff, dtype):
    g = graph_of(s, cutoff, dtype)
    ref, t_ref = run(model, g, False)
    out, t_fast = run(model, g, True)
    rel = lambda a, b: float((a - b).abs().max() / b.abs().max())
    row = {"system": name, "dtype": str(dtype)[6:], "atoms": g.num_nodes, "edges": int(g.edge_index.shape[1]),
           "ref ms": 1e3 * t_ref, "fast ms": 1e3 * t_fast, "speedup": t_ref / t_fast,
           "dE/E": rel(out["energy"], ref["energy"]), "dF/|F|": rel(out["forces"], ref["forces"]),
           "dS/|S|": rel(out["stress"], ref["stress"])}
    print(f"{name:22s} {row['dtype']:8s} {row['atoms']:6d} atoms {row['edges']:8d} edges | "
          f"ref {row['ref ms']:7.1f} ms  fast {row['fast ms']:6.1f} ms ({row['speedup']:4.1f}x) | "
          f"dE/E {row['dE/E']:.1e}  dF {row['dF/|F|']:.1e}  dS {row['dS/|S|']:.1e}")
    return row

## 2. Energy, forces and stress

Three models, each in float64 and float32, on systems of growing size: MACE-OFF23 (small) and a
NequIP on liquid water, MACE-MP-0 (medium) on silicon. NequIP is built with random weights (the
fast path does not depend on what the weights are); the MACE models are the published foundation
models from the hub.

In [3]:
def nequip():
    cfg = from_dict({"model": {"name": "nequip", "cutoff": 5.0, "n_features": 32, "n_interactions": 3,
                               "l_max": 2, "species": [1, 8], "avg_num_neighbors": 40.0}})
    torch.manual_seed(0)
    return ForceStressOutput(build_model(cfg.model), compute_stress=True), 5.0


def foundation(name):
    model = from_pretrained(name, dispersion=False, compute_stress=True, dtype=torch.float64)
    return model, float(model.cutoff)


CASES = [("mace-off23-small", lambda: foundation("mace-off23-small"), [water(n) for n in (4, 8, 12)]),
         ("nequip (water)", nequip, [water(n) for n in (4, 8, 12)]),
         ("mace-mp-0-medium", lambda: foundation("mace-mp-0-medium"),
          [supercell(SILICON, n, 0.05) for n in (3, 4, 8)])]
rows = []
for name, make, systems in CASES:
    for dtype in (torch.float64, torch.float32):
        model, cutoff = make()
        model = model.to(device=device, dtype=dtype).eval()
        for s in systems:
            rows.append(compare(name, model, s, cutoff, dtype))
        del model
        torch.cuda.empty_cache()

mace-off23-small is distributed under the Academic Software License (https://github.com/gabor1/ASL); by using it you accept its terms (no commercial use).


mace-off23-small       float64     192 atoms     7768 edges | ref    15.9 ms  fast   21.6 ms ( 0.7x) | dE/E 0.0e+00  dF 7.8e-14  dS 2.9e-14


mace-off23-small       float64    1536 atoms    61554 edges | ref    44.2 ms  fast   23.1 ms ( 1.9x) | dE/E 2.2e-16  dF 6.4e-14  dS 1.0e-14


mace-off23-small       float64    5184 atoms   208112 edges | ref   141.9 ms  fast   38.3 ms ( 3.7x) | dE/E 3.4e-15  dF 4.7e-14  dS 2.7e-14


mace-off23-small       float32     192 atoms     7768 edges | ref    16.0 ms  fast   22.1 ms ( 0.7x) | dE/E 0.0e+00  dF 2.0e-06  dS 3.0e-06


...

mace-mp-0-medium       float64     512 atoms    22744 edges | ref    60.5 ms  fast   35.6 ms ( 1.7x) | dE/E 5.0e-16  dF 1.1e-15  dS 6.6e-15


mace-mp-0-medium       float64    4096 atoms   182192 edges | ref   447.9 ms  fast   57.3 ms ( 7.8x) | dE/E 6.7e-16  dF 1.3e-15  dS 4.7e-15


mace-mp-0-medium       float32     216 atoms     9600 edges | ref    22.9 ms  fast   35.5 ms ( 0.6x) | dE/E 0.0e+00  dF 3.1e-07  dS 5.2e-07


mace-mp-0-medium       float32     512 atoms    22744 edges | ref    45.3 ms  fast   35.5 ms ( 1.3x) | dE/E 2.7e-07  dF 3.5e-07  dS 5.0e-07


mace-mp-0-medium       float32    4096 atoms   182192 edges | ref   326.5 ms  fast   43.0 ms ( 7.6x) | dE/E 3.6e-07  dF 3.5e-07  dS 5.1e-07


The kernels have a fixed cost per evaluation, so they lose on the smallest systems and win from a
few hundred atoms on. `use_fast="auto"` (the default) takes them above a measured edge count per
GPU and precision (`xnn.gnn.fast.AUTO_POLICY`); for this GPU:

In [4]:
for dtype in (torch.float32, torch.float64):
    print(f"{str(dtype)[6:]}: fast paths from {AUTO_POLICY.threshold(device, dtype):,} edges")
for r in rows:
    picked = AUTO_POLICY.decide(device, getattr(torch, r["dtype"]), r["edges"])
    print(f"  {r['system']:22s} {r['dtype']:8s} {r['atoms']:6d} atoms: auto -> {'fast' if picked else 'reference'}"
          f" (fast path {max(r['speedup'], 1 / r['speedup']):.1f}x {'faster' if r['speedup'] > 1 else 'slower'})")

float32: fast paths from 50,000 edges
float64: fast paths from 25,000 edges
  mace-off23-small       float64     192 atoms: auto -> reference (fast path 1.4x slower)
  mace-off23-small       float64    1536 atoms: auto -> fast (fast path 1.9x faster)
  mace-off23-small       float64    5184 atoms: auto -> fast (fast path 3.7x faster)
  mace-off23-small       float32     192 atoms: auto -> reference (fast path 1.4x slower)
  mace-off23-small       float32    1536 atoms: auto -> fast (fast path 1.6x faster)
  mace-off23-small       float32    5184 atoms: auto -> fast (fast path 3.7x faster)
  nequip (water)         float64     192 atoms: auto -> reference (fast path 1.4x faster)
  nequip (water)         float64    1536 atoms: auto -> fast (fast path 5.6x faster)
  nequip (water)         float64    5184 atoms: auto -> fast (fast path 7.3x faster)
  nequip (water)         float32     192 atoms: auto -> reference (fast path 1.3x faster)
  nequip (water)         float32    1536 atoms: auto -

The thresholds are a compromise across models, since a heavier model reaches the crossover earlier:
on an A100 in float32 it is near 19k edges for MACE-MP-0 medium and near 90k for the light water
models. So `"auto"` gives up a little speed on mid-sized systems of the heavier models (MACE-MP-0
at 512 atoms, the NequIP above at 192 atoms) and never takes the kernels where they lose by
much. `use_fast=True` takes them whatever the size.

The NequIP above is built with the float32 default and cast to float64. Its coupling constants
(Wigner-3j tensors, computed at construction) are rebuilt exactly on the cast, so it agrees with
the fast path to the same ~1e-15 as the MACE models, which load directly in float64.

## 3. Training: gradients of a force-matching loss

The fast paths are differentiable twice, so a force-matching loss trains through them. The
gradient of $\sum E^2 + \sum F^2$ with respect to every parameter, fast against reference, in
float64 (MACE-OFF23 small on 64 waters):

In [5]:
model, cutoff = foundation("mace-off23-small")
model = model.to(device=device, dtype=torch.float64).train()
g = graph_of(water(4), cutoff, torch.float64)
grads = {}
for use_fast in (False, True):
    fast.set_use_fast(model, use_fast)
    model.zero_grad(set_to_none=True)
    out = model(copy.copy(g))
    (out["energy"].square().sum() + out["forces"].square().sum()).backward()
    grads[use_fast] = {n: p.grad.clone() for n, p in model.named_parameters() if p.grad is not None}
errors = {n: float((grads[True][n] - g0).abs().max() / g0.abs().max()) for n, g0 in grads[False].items()}
worst = max(errors, key=errors.get)
print(f"{len(errors)} parameters, worst relative difference {errors[worst]:.1e} ({worst})")

27 parameters, worst relative difference 4.0e-13 (model.interactions.1.linear_up.weight)


## 4. Export

TorchScript and LAMMPS export script the reference implementation, whatever the model ran
before, so an exported model runs wherever TorchScript does. After a fast evaluation:

In [6]:
import tempfile
from xnn.common.deploy import export_torchscript_potential

model.eval()
fast.set_use_fast(model, True)
eager = model(copy.copy(g))
with tempfile.TemporaryDirectory() as tmp:
    path = export_torchscript_potential(model.model, cutoff, f"{tmp}/model.pt")
    scripted = torch.jit.load(path, map_location=device)
print("exported:", type(scripted).__name__)

exported: RecursiveScriptModule


## 5. Summary

In [7]:
cols = ["system", "dtype", "atoms", "speedup", "dE/E", "dF/|F|", "dS/|S|"]
print("".join(f"{c:>18s}" for c in cols))
for r in rows:
    print("".join(f"{r[c]:>18}" if isinstance(r[c], (str, int)) else
                  (f"{r[c]:>18.2f}" if c == "speedup" else f"{r[c]:>18.1e}") for c in cols))

            system             dtype             atoms           speedup              dE/E            dF/|F|            dS/|S|
  mace-off23-small           float64               192              0.73           0.0e+00           7.8e-14           2.9e-14
  mace-off23-small           float64              1536              1.91           2.2e-16           6.4e-14           1.0e-14
  mace-off23-small           float64              5184              3.70           3.4e-15           4.7e-14           2.7e-14
  mace-off23-small           float32               192              0.72           0.0e+00           2.0e-06           3.0e-06
  mace-off23-small           float32              1536              1.62           2.3e-07           2.2e-06           1.8e-06
  mace-off23-small           float32              5184              3.68           2.8e-07           1.1e-06           3.3e-06
    nequip (water)           float64               192              1.41           5.4e-15           1.2e-15   